In [2]:
!pip install -q lightgbm sentence-transformers

In [1]:
# --------- Imports ----------
import os, re, math, gc, time
import numpy as np, pandas as pd
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
from sentence_transformers import SentenceTransformer
from google.colab import files
import warnings
warnings.filterwarnings("ignore")

# ====== SETTINGS ======
USE_SBERT = True        # set False to skip SBERT (faster)
USE_IMAGES = False      # set True to extract image embeddings (very slow unless you have GPU and many minutes)
TFIDF_MAX_FEATURES = 20000
SBERT_PCA_DIM = 128     # reduce SBERT dim to save memory
NFOLD = 5
RANDOM_STATE = 42

In [3]:
print("Upload student_resource.zip (you'll be prompted).")
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]
print("Uploaded:", zip_path)

Upload student_resource.zip (you'll be prompted).


Saving 68e8d1d70b66d_student_resource.zip to 68e8d1d70b66d_student_resource.zip
Uploaded: 68e8d1d70b66d_student_resource.zip


In [4]:
!unzip -q -o "{zip_path}" -d /content/
BASE = "/content/student_resource"
TRAIN_PATH = os.path.join(BASE, "dataset", "train.csv")
TEST_PATH  = os.path.join(BASE, "dataset", "test.csv")
print("Loading data from:", TRAIN_PATH, TEST_PATH)
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
print("Train:", train.shape, "Test:", test.shape)

Loading data from: /content/student_resource/dataset/train.csv /content/student_resource/dataset/test.csv
Train: (75000, 4) Test: (75000, 3)


In [6]:
def extract_ipq(text):
    if not isinstance(text, str): return np.nan
    t = text.lower()
    patterns = [r'pack of\s*(\d+)', r'(\d+)\s*[-]?\s*pack', r'(\d+)\s*count\b',
                r'(\d+)\s*pcs\b', r'(\d+)\s*pieces\b', r'(\d+)\s*x\b']
    for pat in patterns:
        m = re.search(pat, t)
        if m:
            try: return int(m.group(1))
            except: pass
    return np.nan

def extract_brand(title):
    if not isinstance(title, str) or title.strip()=="":
        return "unknown"
    parts = re.split(r'[-:|]', title, maxsplit=1)
    first = parts[0].strip().split()[0]
    first = re.sub(r'[^A-Za-z0-9]', '', first)
    if len(first) <= 1:
        return "unknown"
    return first.lower()

def extract_unit_amount(text):
    if not isinstance(text, str): return (np.nan, None)
    t = text.lower()
    m = re.search(r'(\d+[\.,]?\d*)\s*(kg|g|gm|grams|ml|l|litre|ltr|oz|pack|pcs|pieces)', t)
    if not m: return (np.nan, None)
    num = float(m.group(1).replace(',', '.'))
    unit = m.group(2)
    if unit in ['kg']: return (num * 1000, 'g')
    if unit in ['g','gm','grams']: return (num, 'g')
    if unit in ['l','litre','ltr']: return (num * 1000, 'ml')
    if unit in ['ml']: return (num, 'ml')
    if unit in ['oz']: return (num * 28.3495, 'g')
    if unit in ['pack','pcs','pieces']: return (num, 'pack')
    return (num, unit)

In [7]:
for df in (train, test):
    df['catalog_content'] = df['catalog_content'].fillna('').astype(str)
    # simple title heuristics
    df['title'] = df['catalog_content'].apply(lambda x: x.split(' - ')[0].split('|')[0][:200])
    df['ipq'] = df['catalog_content'].apply(extract_ipq)
    df['brand'] = df['title'].apply(extract_brand)
    units = df['catalog_content'].apply(extract_unit_amount)
    df['unit_amount'] = units.apply(lambda x: x[0])
    df['unit_type'] = units.apply(lambda x: x[1])
    df['has_image'] = df['image_link'].notnull().astype(int)
    df['title_has_pack'] = df['catalog_content'].str.contains(r'pack|bundle|combo', flags=re.I).astype(int)
    df['title_has_refill'] = df['catalog_content'].str.contains(r'refill', flags=re.I).astype(int)
    df['text'] = (df['title'].fillna('') + " " + df['catalog_content'].fillna(''))


In [8]:
# price_per_unit (train only where available)
train['price_per_unit'] = np.nan
mask = train['unit_amount'].notnull() & (train['unit_amount'] > 0)
train.loc[mask, 'price_per_unit'] = train.loc[mask, 'price'] / train.loc[mask, 'unit_amount']
median_ppu = train['price_per_unit'].median()
print("Median price_per_unit (train):", median_ppu)

Median price_per_unit (train): 0.057724416068478585


In [9]:
# ====== TF-IDF pipeline ======
print("Building TF-IDF...")
tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1,2), min_df=5)
tfidf.fit(pd.concat([train['text'], test['text']]))
X_text_train = tfidf.transform(train['text'])
X_text_test  = tfidf.transform(test['text'])
print("TF-IDF shapes:", X_text_train.shape, X_text_test.shape)

Building TF-IDF...
TF-IDF shapes: (75000, 20000) (75000, 20000)


In [10]:
# ====== SBERT embeddings (optional but high impact) ======
if USE_SBERT:
    print("Computing SBERT embeddings (this may take a few minutes)...")
    sbert = SentenceTransformer('all-MiniLM-L6-v2')  # compact & fast
    train_texts = train['text'].fillna('').tolist()
    test_texts  = test['text'].fillna('').tolist()
    emb_train = sbert.encode(train_texts, batch_size=64, show_progress_bar=True)
    emb_test  = sbert.encode(test_texts, batch_size=64, show_progress_bar=True)
    print("SBERT raw dims:", emb_train.shape, emb_test.shape)
    # reduce dims with PCA for memory/backbone speed
    print(f"Reducing SBERT dim -> {SBERT_PCA_DIM} via PCA")
    pca = PCA(n_components=SBERT_PCA_DIM, random_state=RANDOM_STATE)
    emb_train_red = pca.fit_transform(emb_train)
    emb_test_red  = pca.transform(emb_test)
    # scale embeddings
    scaler_emb = StandardScaler()
    emb_train_scaled = scaler_emb.fit_transform(emb_train_red)
    emb_test_scaled  = scaler_emb.transform(emb_test_red)
    del emb_train, emb_test, emb_train_red, emb_test_red
    gc.collect()
else:
    emb_train_scaled = None
    emb_test_scaled  = None

Computing SBERT embeddings (this may take a few minutes)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

SBERT raw dims: (75000, 384) (75000, 384)
Reducing SBERT dim -> 128 via PCA


In [11]:
if USE_IMAGES:
    print("Image embeddings enabled — this may run for a long time. Ensure GPU runtime.")
    try:
        # Try to import provided downloader
        sys.path.append(os.path.join(BASE, "src"))
        from utils import download_images  # if provided in src/utils.py
        img_dir = "/content/images"
        os.makedirs(img_dir, exist_ok=True)
        print("Downloading images (may be throttled)...")
        download_images(train, img_dir, id_col='sample_id', url_col='image_link', max_workers=8)
        download_images(test, img_dir, id_col='sample_id', url_col='image_link', max_workers=8)
    except Exception as e:
        print("No downloader helper found or failed; you can implement image download yourself.", e)
    # Use EfficientNetB0 to create pooled embeddings
    import tensorflow as tf
    from tensorflow.keras.applications import EfficientNetB0
    from tensorflow.keras.applications.efficientnet import preprocess_input
    from tensorflow.keras.preprocessing import image
    model_img = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')
    def img_path_for_id(sid):
        for ext in ['.jpg','.png','.jpeg','.webp']:
            p = f"/content/images/{sid}{ext}"
            if os.path.exists(p): return p
        return None
    def get_img_emb(sid):
        p = img_path_for_id(sid)
        if p is None: return np.zeros((model_img.output_shape[1],), dtype=float)
        try:
            img = image.load_img(p, target_size=(224,224))
            arr = image.img_to_array(img)
            arr = np.expand_dims(arr,0)
            arr = preprocess_input(arr)
            emb = model_img.predict(arr, verbose=0)[0]
            return emb
        except:
            return np.zeros((model_img.output_shape[1],), dtype=float)
    # Build embeddings arrays (careful — memory heavy)
    img_emb_train = np.vstack([get_img_emb(sid) for sid in train['sample_id']])
    img_emb_test  = np.vstack([get_img_emb(sid) for sid in test['sample_id']])
    # reduce dims
    pca_img = PCA(n_components=64, random_state=RANDOM_STATE)
    img_train_red = pca_img.fit_transform(img_emb_train)
    img_test_red  = pca_img.transform(img_emb_test)
    # scale
    scaler_img = StandardScaler()
    img_train_scaled = scaler_img.fit_transform(img_train_red)
    img_test_scaled  = scaler_img.transform(img_test_red)
    del img_emb_train, img_emb_test, img_train_red, img_test_red
    gc.collect()
else:
    img_train_scaled = None
    img_test_scaled  = None


In [12]:
# ====== Numeric frame for both models ======
num_cols = ['ipq','has_image','title_has_pack','title_has_refill']
X_num_train = pd.DataFrame({
    'ipq': train['ipq'].fillna(1).astype(float),
    'has_image': train['has_image'].astype(int),
    'title_has_pack': train['title_has_pack'].astype(int),
    'title_has_refill': train['title_has_refill'].astype(int),
})
X_num_test = pd.DataFrame({
    'ipq': test['ipq'].fillna(1).astype(float),
    'has_image': test['has_image'].astype(int),
    'title_has_pack': test['title_has_pack'].astype(int),
    'title_has_refill': test['title_has_refill'].astype(int),
})
# placeholder for brand target encoding (will fill per-fold)
X_num_train['brand_te'] = 0.0
X_num_test['brand_te'] = 0.0

# If we have SBERT or images, append them later into numeric/dense matrices.

In [23]:
# ====== CV function to train a LightGBM model given features (sparse or dense) ======
def run_kfold_lgb(X_sparse, X_dense, y_log, test_sparse, test_dense, lgb_params, nf=NFOLD):
    """
    X_sparse: scipy.sparse or None (TF-IDF)
    X_dense: np.array or DataFrame or None (SBERT, numeric)
    Returns: oof_preds_log (train), test_preds_log
    """
    n = len(y_log)
    oof = np.zeros(n)
    test_preds = np.zeros(test_sparse.shape[0] if X_sparse is not None else test_dense.shape[0])
    kf = KFold(n_splits=nf, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(range(n))):
        print(f"Fold {fold+1}/{nf}")
        # in-fold brand encoding
        tr = train.iloc[tr_idx]; val = train.iloc[val_idx]
        brand_map = tr.groupby('brand')['price'].median()
        # fill brand_te in X_num_train for validation indices
        X_num_train.loc[val_idx, 'brand_te'] = val['brand'].map(brand_map).fillna(tr['price'].median()).values
        # compute full data brand map for test
        full_brand_map = train.groupby('brand')['price'].median()
        X_num_test['brand_te'] = test['brand'].map(full_brand_map).fillna(train['price'].median()).values

        # build sparse/dense for this fold
        part_sparse_tr = X_sparse[tr_idx] if X_sparse is not None else None
        part_sparse_val = X_sparse[val_idx] if X_sparse is not None else None
        part_sparse_test = test_sparse if X_sparse is not None else None

        # Dense: concatenate numeric + optional SBERT + optional images
        dense_tr_parts = [X_num_train.loc[tr_idx].values]
        dense_val_parts = [X_num_train.loc[val_idx].values]
        dense_test_parts = [X_num_test.values]
        if X_dense is not None:
            dense_tr_parts.append(X_dense[tr_idx])
            dense_val_parts.append(X_dense[val_idx])
            dense_test_parts.append(test_dense)  # test_dense must correspond
        Dtr = np.hstack(dense_tr_parts)
        Dval = np.hstack(dense_val_parts)
        Dtest = np.hstack(dense_test_parts)

        # Combine sparse + dense into final train/val/test
        if part_sparse_tr is not None:
            Xtr = hstack([part_sparse_tr, csr_matrix(Dtr)])
            Xval = hstack([part_sparse_val, csr_matrix(Dval)])
            Xte  = hstack([part_sparse_test, csr_matrix(Dtest)])
        else:
            # use dense only -> convert to csr to feed into LGB
            Xtr = csr_matrix(Dtr)
            Xval = csr_matrix(Dval)
            Xte  = csr_matrix(Dtest)


        ytr = y_log[tr_idx]; yval = y_log[val_idx]

        dtrain = lgb.Dataset(Xtr, label=ytr)
        dval = lgb.Dataset(Xval, label=yval)

        model = lgb.train(lgb_params, dtrain, num_boost_round=3000,
                          valid_sets=[dtrain, dval], callbacks=[lgb.early_stopping(100)])


        oof[val_idx] = model.predict(Xval, num_iteration=model.best_iteration)
        test_preds += model.predict(Xte, num_iteration=model.best_iteration) / nf

        # free memory
        del Xtr, Xval, dtrain, dval, model
        gc.collect()

    return oof, test_preds

In [24]:
# ====== Prepare targets and LGB params ======
y = train['price'].values
y_log = np.log1p(y)

lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,
    'num_leaves': 64,
    'min_data_in_leaf': 40,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'verbosity': -1,
    'seed': RANDOM_STATE
}

In [30]:
# ------------------- FAST MODE (paste into Colab) -------------------
# Fast-mode settings (small sacrifice in final SMAPE for big time savings)
USE_SBERT = False            # skip sentence-transformers (set True later)
TFIDF_MAX_FEATURES = 5000    # much smaller
NFOLD = 3                    # fewer folds -> faster
BOOST_ROUNDS = 800           # fewer rounds
EARLY_STOP = 50              # earlier stop
TRUNC_SVD_DIM = 300          # set to None to skip SVD compression

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
import gc, time

# Rebuild / reduce TF-IDF if required (overwrite previous tfidf)
print("Rebuilding TF-IDF with", TFIDF_MAX_FEATURES, "features...")
tfidf = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1,2), min_df=5)
tfidf.fit(pd.concat([train['text'], test['text']]))
X_text_train = tfidf.transform(train['text'])
X_text_test  = tfidf.transform(test['text'])
print("Shapes:", X_text_train.shape, X_text_test.shape)

# Optional: compress TF-IDF -> low-rank dense with TruncatedSVD (speeds up LGB)
if TRUNC_SVD_DIM is not None:
    print("Applying TruncatedSVD ->", TRUNC_SVD_DIM)
    svd = TruncatedSVD(n_components=TRUNC_SVD_DIM, random_state=42)
    X_text_train_svd = svd.fit_transform(X_text_train)
    X_text_test_svd  = svd.transform(X_text_test)
    # scale SVD outputs if you like (helps LGB)
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler()
    X_text_train_svd = sc.fit_transform(X_text_train_svd)
    X_text_test_svd  = sc.transform(X_text_test_svd)
    # convert to dense arrays for the dense-only run
    X_sparse_for_model = None
    X_dense_text_train = X_text_train_svd
    X_dense_text_test  = X_text_test_svd
    del X_text_train, X_text_test
    gc.collect()
else:
    X_sparse_for_model = X_text_train
    X_dense_text_train = None
    X_dense_text_test  = None

# numeric frames (reuse existing X_num_train/test)
# ensure brand_te placeholder present
if 'brand_te' not in X_num_train.columns:
    X_num_train['brand_te'] = 0.0
    X_num_test['brand_te']  = 0.0

# Adjust LGB params for faster training (a bit more regular)
lgb_params_fast = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'min_data_in_leaf': 40,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbosity': -1,
    'seed': 42,
    'n_jobs': 4
}

# A light wrapper to train quickly using fewer rounds and folds
def run_kfold_fast(X_sparse, X_dense_text, X_dense_other, y_log, test_sparse, test_dense_text, test_dense_other):
    n = len(y_log)
    oof = np.zeros(n)
    test_preds = np.zeros(len(test))
    kf = KFold(n_splits=NFOLD, shuffle=True, random_state=42)
    for fold, (tr_idx, val_idx) in enumerate(kf.split(range(n))):
        print("Fold", fold+1, "of", NFOLD)
        tr = train.iloc[tr_idx]; val = train.iloc[val_idx]
        # in-fold brand encoding
        brand_map = tr.groupby('brand')['price'].median()
        X_num_train.loc[val_idx, 'brand_te'] = val['brand'].map(brand_map).fillna(tr['price'].median()).values
        full_brand_map = train.groupby('brand')['price'].median()
        X_num_test['brand_te'] = test['brand'].map(full_brand_map).fillna(train['price'].median()).values

        # Build features: combine sparse/dense parts
        # If we compressed TF-IDF to dense (X_dense_text), use that; otherwise use sparse
        if X_dense_text is not None:
            Dtr_text = X_dense_text[tr_idx]; Dval_text = X_dense_text[val_idx]; Dtest_text = test_dense_text
            Dtr = np.hstack([X_num_train.loc[tr_idx].values, Dtr_text])
            Dval = np.hstack([X_num_train.loc[val_idx].values, Dval_text])
            Dtest = np.hstack([X_num_test.values, Dtest_text])
            Xtr = csr_matrix(Dtr); Xval = csr_matrix(Dval); Xte = csr_matrix(Dtest)
        else:
            # sparse text + numeric
            Xtr = hstack([X_sparse[tr_idx], csr_matrix(X_num_train.loc[tr_idx].values)])
            Xval = hstack([X_sparse[val_idx], csr_matrix(X_num_train.loc[val_idx].values)])
            Xte  = hstack([test_sparse, csr_matrix(X_num_test.values)])

        ytr = y_log[tr_idx]; yval = y_log[val_idx]
        dtrain = lgb.Dataset(Xtr, label=ytr)
        dval = lgb.Dataset(Xval, label=yval)
        model = lgb.train(lgb_params_fast, dtrain, num_boost_round=BOOST_ROUNDS,
                          valid_sets=[dtrain, dval], callbacks=[lgb.early_stopping(EARLY_STOP)])
        oof[val_idx] = model.predict(Xval, num_iteration=model.best_iteration)
        test_preds += model.predict(Xte, num_iteration=model.best_iteration) / NFOLD
        del Xtr, Xval, dtrain, dval, model
        gc.collect()
    return oof, test_preds

# Choose which features to pass to fast runner
if X_dense_text_train is not None:
    # SVD compressed TF-IDF (dense) pipeline
    X_sparse_pass = None
    X_dense_text = X_dense_text_train
    test_dense_text = X_dense_text_test
else:
    X_sparse_pass = X_sparse_for_model
    X_dense_text = None
    test_dense_text = None

print("Running fast K-Fold training...")
t0 = time.time()
oof_fast_log, test_fast_log = run_kfold_fast(
    X_sparse = X_sparse_pass,
    X_dense_text = X_dense_text,
    X_dense_other = None,
    y_log = y_log,
    test_sparse = X_text_test if X_sparse_pass is not None else None,
    test_dense_text = test_dense_text,
    test_dense_other = None
)
t1 = time.time()
print("Fast training done in {:.1f} min".format((t1-t0)/60.0))

oof_fast_price = np.expm1(oof_fast_log)
test_fast_price = np.expm1(test_fast_log)
# Need to import smape function
# print("Fast-mode OOF SMAPE:", smape(y, oof_fast_price))

# Save quick submission
out_fast = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.maximum(test_fast_price, 0.01)})
out_fast.to_csv("test_out_fast.csv", index=False, float_format="%.6f")
from google.colab import files
files.download("test_out_fast.csv")
# ------------------- END FAST MODE -------------------

Rebuilding TF-IDF with 5000 features...
Shapes: (75000, 5000) (75000, 5000)
Applying TruncatedSVD -> 300
Running fast K-Fold training...
Fold 1 of 3
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[799]	training's rmse: 0.370439	valid_1's rmse: 0.72995
Fold 2 of 3
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[800]	training's rmse: 0.372639	valid_1's rmse: 0.720283
Fold 3 of 3
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[800]	training's rmse: 0.372128	valid_1's rmse: 0.724158
Fast training done in 5.5 min


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>